In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 复用之前的 SwiGLU FFN
class SwiGLUExpert(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.gate_proj = nn.Linear(dim, hidden_dim, bias=False)
        self.up_proj = nn.Linear(dim, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, dim, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

class DeepSeekMoE(nn.Module):
    def __init__(
        self,
        dim: int,
        hidden_dim: int,
        num_experts: int,        # 总路由专家数 (例如 64)
        num_shared_experts: int, # 共享专家数 (例如 2)
        top_k: int,              # 每次激活多少个路由专家 (例如 6)
    ):
        super().__init__()
        self.dim = dim
        self.num_experts = num_experts
        self.num_shared_experts = num_shared_experts
        self.top_k = top_k

        # --- 1. 共享专家 (Always On) ---
        # 负责捕获通用知识，对所有 Token 都激活
        self.shared_experts = nn.ModuleList([
            SwiGLUExpert(dim, hidden_dim) for _ in range(num_shared_experts)
        ])

        # --- 2. 路由专家 (Routed/Fine-Grained) ---
        # 负责捕获稀疏/专业知识，只对特定 Token 激活
        self.routed_experts = nn.ModuleList([
            SwiGLUExpert(dim, hidden_dim) for _ in range(num_experts)
        ])

        # --- 3. 路由器 (Router) ---
        # 很多论文建议对 Router 的输入进行归一化以稳定训练
        self.router = nn.Linear(dim, num_experts, bias=False)

    def forward(self, x):
        # x shape: (Batch, Seq_Len, Dim)
        batch_size, seq_len, dim = x.shape
        # 展平以便统一处理: (B*L, D)
        x_flat = x.view(-1, dim)
        
        # ==========================================
        # Part A: 共享专家前向传播 (Shared Path)
        # ==========================================
        shared_output = 0
        if self.num_shared_experts > 0:
            # 多个共享专家的输出通常直接求和或取平均
            for expert in self.shared_experts:
                shared_output += expert(x_flat)
        
        # ==========================================
        # Part B: 路由专家前向传播 (Routed Path)
        # ==========================================
        
        # 1. 计算路由分数
        # DeepSeek 常用 trick: 对 router 输入做 normalize
        router_logits = self.router(F.normalize(x_flat, dim=-1)) # (B*L, N_experts)
        
        # 2. Top-K 选择
        # indices: 每个 token 选中了哪 k 个专家
        # weights: 对应的权重
        weights, indices = torch.topk(router_logits, self.top_k, dim=-1)
        
        # 3. Softmax 归一化 (只对选中的 k 个做 softmax)
        weights = F.softmax(weights, dim=-1)
        
        # 4. 专家计算 (Naive 循环实现，易于理解)
        # 生产环境通常使用 torch.scatter_add 或 Triton kernel 优化
        
        routed_output = torch.zeros_like(x_flat)
        
        # 遍历每一个专家，找到选中它的 token 并计算
        # 这种写法显存并不高效，但逻辑最清晰
        for i, expert in enumerate(self.routed_experts):
            # 创建掩码：哪些 token 的 Top-K 中包含了当前专家 i
            # indices shape: (Total_Tokens, TopK)
            # mask shape: (Total_Tokens)
            # any(dim=1) 意味着只要 TopK 里有 i，这个 token 就要过这个专家
            batch_mask = (indices == i).any(dim=-1)
            
            if batch_mask.any():
                # 选出需要该专家的 token
                selected_tokens = x_flat[batch_mask]
                
                # 专家计算
                expert_out = expert(selected_tokens)
                
                # 找到对应的权重
                # 我们需要知道当前专家 i 是该 token 的第几个选择 (rank)，以便取对应的 weight
                # (indices == i) -> (Total_Tokens, TopK) boolean matrix
                # nonzero() -> 拿到坐标
                
                # 为了简单演示加权求和，这里简化处理：
                # 我们直接构造一个全量的 weight 矩阵
                # 实际高效实现会用到 sparse operations
                
                # 简单做法：
                # 1. 把 expert_out 乘上对应的权重
                # 2. 加回 routed_output
                
                # 获取权重: (Num_Selected_Tokens, )
                # 这是一个稍微 trick 的取法
                row_indices = torch.where(batch_mask)[0]
                # 找到 i 在 indices 中的位置 (0~k-1)
                col_indices = torch.where(indices[batch_mask] == i)[1]
                selected_weights = weights[batch_mask, col_indices]
                
                # 加权: Expert_Out * Weight
                weighted_expert_out = expert_out * selected_weights.unsqueeze(-1)
                
                # 累加回主输出
                routed_output[row_indices] += weighted_expert_out

        # ==========================================
        # Part C: 结果融合
        # ==========================================
        # DeepSeekMoE: Output = Shared_Output + Routed_Output
        final_output = shared_output + routed_output
        
        return final_output.view(batch_size, seq_len, dim)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass, field
from typing import Tuple, List, Union

# ==========================================
# 1. 基础组件：SwiGLU Expert
# ==========================================
class SwiGLUExpert(nn.Module):
    def __init__(self, args):
        super().__init__()
        # 使用 args.moe_ffn_hidden_size (768) 而非 args.ffn_hidden_size (4096)
        # 遵循 args.disable_bias_linear (True -> bias=False)
        dim = args.hidden_size
        hidden_dim = args.moe_ffn_hidden_size
        use_bias = not args.disable_bias_linear

        self.gate_proj = nn.Linear(dim, hidden_dim, bias=use_bias)
        self.up_proj = nn.Linear(dim, hidden_dim, bias=use_bias)
        self.down_proj = nn.Linear(hidden_dim, dim, bias=use_bias)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

# ==========================================
# 2. Standard Top-K MoE Layer
# ==========================================
class TopKMoELayer(nn.Module):
    def __init__(self, args):
        super().__init__()
        self.args = args
        self.num_experts = args.num_experts
        self.top_k = args.moe_router_topk
        self.aux_loss_coeff = args.moe_aux_loss_coeff

        # --- 1. 专家池 ---
        # 纯路由专家，忽略 shared expert 参数
        self.experts = nn.ModuleList([
            SwiGLUExpert(args) for _ in range(self.num_experts)
        ])

        # --- 2. 路由器 (Router) ---
        # 遵循 args.moe_router_enable_expert_bias
        self.router = nn.Linear(
            args.hidden_size, 
            args.num_experts, 
            bias=args.moe_router_enable_expert_bias
        )

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        x: (Batch, Seq_Len, Hidden_Size)
        """
        batch_size, seq_len, hidden_dim = x.shape
        x_flat = x.view(-1, hidden_dim)

        # ==========================================
        # Router 计算
        # ==========================================
        # 1. 强制 fp32 计算 Router (即便模型是 bf16)
        router_input = x_flat.float()
        
        # 2. Normalize (常用 trick，尽管 config 没显式写，但通常配合 Sigmoid 使用)
        router_input = F.normalize(router_input, dim=-1)
        
        # 3. Linear Projection
        logits = self.router(router_input)

        # 4. Score Function (Sigmoid vs Softmax)
        if self.args.moe_router_score_function == 'sigmoid':
            routing_weights = torch.sigmoid(logits)
        else:
            routing_weights = F.softmax(logits, dim=-1)

        # ==========================================
        # Top-K 选择
        # ==========================================
        # weights: (B*L, k)
        # indices: (B*L, k)
        selected_weights, selected_indices = torch.topk(routing_weights, self.top_k, dim=-1)

        # 5. 归一化选中的权重 (Renormalize selected weights to sum to 1)
        # 即使是 Sigmoid，为了保持输出幅度一致，通常也会对选中的 TopK 进行归一化
        selected_weights = selected_weights / (selected_weights.sum(dim=-1, keepdim=True) + 1e-6)

        # ==========================================
        # 辅助损失 (Aux Loss)
        # ==========================================
        aux_loss = 0.0
        if self.training and self.args.moe_router_load_balancing_type == 'seq_aux_loss':
            # 简单的 Load Balancing Loss
            # mask: (Tokens, Experts) 
            mask = torch.zeros_like(routing_weights).scatter_(1, selected_indices, 1.0)
            density = mask.mean(dim=0) # 专家实际上被选中的概率
            density_proxy = routing_weights.mean(dim=0) # 路由器的输出概率 (可导)
            
            aux_loss = (self.aux_loss_coeff * self.num_experts) * (density * density_proxy).sum()

        # ==========================================
        # 专家分发 (Naive Loop)
        # ==========================================
        final_output = torch.zeros_like(x_flat)

        for i, expert in enumerate(self.experts):
            # 找出选中专家 i 的 token
            batch_mask = (selected_indices == i).any(dim=-1)
            
            if batch_mask.any():
                # 提取 Input
                selected_tokens = x_flat[batch_mask]
                
                # Expert 计算
                expert_out = expert(selected_tokens)
                
                # 获取权重: 需要找到专家 i 在 Top-K 中的位置
                # col_indices 是 0 到 k-1 的数值
                col_indices = torch.where(selected_indices[batch_mask] == i)[1]
                weight = selected_weights[batch_mask, col_indices].unsqueeze(-1)
                
                # 加权累加
                # 注意：这里如果用 += 在 inplace 操作可能会影响梯度，严谨实现应用 index_add
                final_output[batch_mask] += expert_out * weight

        return final_output.view(batch_size, seq_len, hidden_dim), aux_loss

In [ ]:
from config import ModelArgs

model_args = ModelArgs()

print(">>> 正在使用 ModelArgs 初始化 StandardTopKMoELayer...")
moe_layer = TopKMoELayer(model_args)

# 验证关键参数
print(f"Experts Count: {len(moe_layer.experts)} (Config: 64)")
print(f"Top K: {moe_layer.top_k} (Config: 4)")
print(f"Expert Hidden Dim: {moe_layer.experts[0].gate_proj.out_features} (Config: 768)")
print(f"Router Bias Enabled: {moe_layer.router.bias is not None} (Config: True)")
print(f"Router Score Function: {model_args.moe_router_score_function} (Config: sigmoid)")

# 运行 Dummy Forward
x = torch.randn(2, 128, 2048) # (Batch, Seq, Hidden)
output, aux_loss = moe_layer(x)

print("\n>>> Forward Pass Successful")
print(f"Input Shape: {x.shape}")
print(f"Output Shape: {output.shape}")
print(f"Aux Loss: {aux_loss}")